# SAE playground — быстрые эксперименты

Сравниваем ответы **без интервенции** и **с SAE latent bump** на одних и тех же промптах. Нужны GPU; для Mistral Instruct часто нужен Hugging Face.

**Код `sae_muc/`** в следующей ячейке **клонируется с GitHub** (по умолчанию репозиторий только с пакетом `sae-muc`). Можно переопределить URL через переменную окружения `SAE_MUC_GIT_URL` или правку `GIT_URL` в ячейке.

Опционально: `sae_muc/artifacts/mistral_intervention.pt`. Без него — **случайный** `delta` (проверка хуков, не VUF).

## 1. Клон репозитория с GitHub + установка зависимостей

In [ ]:
import os, sys, subprocess

# Репозиторий с корнем вида: .../sae_muc/__init__.py (например github.com/SadreevAmir/sae-muc)
GIT_URL = os.environ.get("SAE_MUC_GIT_URL", "https://github.com/SadreevAmir/sae-muc.git")
GIT_BRANCH = os.environ.get("SAE_MUC_BRANCH", "main")
REPO_DIR = os.environ.get("SAE_MUC_REPO_DIR", "/content/sae-muc")

sae_pkg = os.path.join(REPO_DIR, "sae_muc")
if not os.path.isdir(sae_pkg):
    parent = os.path.dirname(REPO_DIR.rstrip("/")) or "/content"
    os.makedirs(parent, exist_ok=True)
    if os.path.isdir(REPO_DIR):
        subprocess.run(["rm", "-rf", REPO_DIR], check=True)
    subprocess.run(
        ["git", "clone", "--depth", "1", "-b", GIT_BRANCH, GIT_URL, REPO_DIR],
        check=True,
    )
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)

assert os.path.isdir(sae_pkg), f"После клона ожидается {sae_pkg}"
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Colab: NumPy 2.0.x — без binary mismatch у transformers; <2.1 тише numba/tensorflow в образе
!pip install -q -U "numpy>=2.0.0,<2.1"
!pip install -q sae-lens transformers accelerate

import torch
assert torch.cuda.is_available(), "Нужен GPU runtime"
print("REPO_DIR:", REPO_DIR)
print("GPU:", torch.cuda.get_device_name(0))

## 2. Конфиг (поменяйте здесь)

In [ ]:
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"

# Слои HF, где висит SAE из релиза mistral-7b-res-wg: 7, 15, 23 (часто хватает 15 и 23)
PROCESS_LAYERS = [15, 23]

ALPHA = 2.0  # сила интервенции (подберите 0.5–5)

# Путь к конфигу из build_intervention_config.py; None = случайный delta для теста железа
INTERVENTION_PT = None  # например: f"{REPO_DIR}/sae_muc/artifacts/mistral_intervention.pt"

SAE_RELEASE = "mistral-7b-res-wg"
SAE_DTYPE = "float32"

## 3. Hugging Face (для gated моделей)

In [ ]:
from huggingface_hub import login
login()  # один раз за сессию

## 4. Загрузка модели и SAE

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from sae_lens import SAE

from sae_muc.hooks import clear_sae_latent_hooks, register_sae_latent_hooks

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16, device_map="auto"
)
model.eval()
model.generation_config.pad_token_id = tokenizer.pad_token_id


def load_intervention_or_random(intervention_path, release, process_layers, sae_dtype):
    layer_to_sae = {}
    layer_to_delta = {}
    if intervention_path and __import__("os").path.isfile(intervention_path):
        blob = torch.load(intervention_path, map_location="cpu")
        release = blob["release"]
        for k, meta in blob["layers"].items():
            hf_layer = int(k)
            if hf_layer not in process_layers:
                continue
            sid = meta["sae_id"]
            print(f"SAE layer {hf_layer} <- {sid}")
            layer_to_sae[hf_layer] = SAE.from_pretrained(
                release, sid, device="cpu", dtype=sae_dtype
            )
            layer_to_delta[hf_layer] = meta["delta"]
    else:
        print("Нет INTERVENTION_PT — случайный delta (только проверка, что хуки работают)")
        from sae_muc.layer_map import hf_layers_for_release
        mapping = {hf: sid for hf, sid in hf_layers_for_release(release)}
        for L in process_layers:
            if L not in mapping:
                continue
            sid = mapping[L]
            sae = SAE.from_pretrained(release, sid, device="cpu", dtype=sae_dtype)
            g = torch.randn(sae.cfg.d_sae, dtype=torch.float32)
            g = g / (g.norm() + 1e-8)
            layer_to_sae[L] = sae
            layer_to_delta[L] = g
    return layer_to_sae, layer_to_delta


layer_to_sae, layer_to_delta = load_intervention_or_random(
    INTERVENTION_PT, SAE_RELEASE, PROCESS_LAYERS, SAE_DTYPE
)
print("Загружено слоёв с SAE:", list(layer_to_sae.keys()))

## 5. Генерация: baseline vs steering

In [ ]:
def build_inputs(user_text: str, system: str | None = None):
    if system:
        messages = [
            {"role": "system", "content": system},
            {"role": "user", "content": user_text},
        ]
    else:
        messages = [{"role": "user", "content": user_text}]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)


@torch.inference_mode()
def generate_one(user_text: str, max_new_tokens: int = 128, temperature: float = 0.7, do_sample: bool = True):
    enc = build_inputs(user_text)
    out = model.generate(
        **enc,
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        temperature=temperature if do_sample else None,
        pad_token_id=tokenizer.pad_token_id,
    )
    gen = out[0, enc["input_ids"].shape[1] :]
    return tokenizer.decode(gen, skip_special_tokens=True).strip()


def compare(user_text: str, alpha: float = None, **gen_kw):
    a = ALPHA if alpha is None else alpha
    clear_sae_latent_hooks(model)
    base = generate_one(user_text, **gen_kw)
    register_sae_latent_hooks(
        model, layer_to_sae, layer_to_delta, PROCESS_LAYERS, float(a)
    )
    steered = generate_one(user_text, **gen_kw)
    clear_sae_latent_hooks(model)
    return base, steered


def show_pair(title: str, user_text: str, alpha: float = None, **gen_kw):
    b, s = compare(user_text, alpha=alpha, **gen_kw)
    a = ALPHA if alpha is None else alpha
    print("=" * 60)
    print(title)
    print("Q:", user_text[:200], "..." if len(user_text) > 200 else "")
    print("-" * 60)
    print("[без SAE]\n", b)
    print("-" * 60)
    print(f"[α={a}, слои {PROCESS_LAYERS}]\n", s)
    print()

## 6. Примеры (замените на свои вопросы)

In [ ]:
show_pair(
    "Факт",
    "What is the 29th largest city in England? Answer in one short phrase.",
    do_sample=False,
    temperature=0.1,
    max_new_tokens=80,
)

show_pair(
    "Осторожная формулировка",
    "Are you completely sure about factual claims? Reply briefly.",
    max_new_tokens=100,
)

## 7. Подбор α (цикл)

In [ ]:
PROMPT = "Name one thing you are uncertain about regarding the year 2150. One sentence."
for a in [0.0, 1.0, 2.0, 4.0]:
    if a == 0.0:
        clear_sae_latent_hooks(model)
        t = generate_one(PROMPT, max_new_tokens=80, do_sample=False)
        print(f"α=0 (baseline): {t!r}")
    else:
        b, s = compare(PROMPT, alpha=a, max_new_tokens=80, do_sample=False)
        print(f"α={a}: {s!r}")